In [1]:
# delta-rs is orders of magnitude faster than Spark for small-medium Delta writes,
# since we skip JVM startup and let Arrow stream the Parquet commits directly.
!pip install --quiet deltalake==0.18.2


StatementMeta(, 224b4d75-d295-42d4-bdc3-7b90622fea45, 3, Finished, Available, Finished, False)

In [13]:
import os, math
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
from deltalake import write_deltalake
import notebookutils

STORE_DATA_TO_BRONZE = 1
STORE_DATA_TO_SILVER = 1
STORE_DATA_TO_GOLD   = 1

NUM_CUSTOMERS = 10_000       # how many customers to generate
NUM_ORDERS    = 50_000       # how many orders to generate (total)

NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION = 5  # split customer writes across N transactions
NUM_ORDER_DELTA_LAKE_TRANSACTION     = 12  # split order writes across N transactions

# Which lakehouse Bronze parquet lands in. Must exist and be visible to this notebook.
BRONZE_LAKEHOUSE_NAME = "bronze"

countries = np.array(["USA", "Canada", "Germany", "France", "UK", "Spain", "Italy", "Sweden"])
cities    = np.array(["New York", "San Francisco", "Toronto", "Berlin", "Paris",
                      "London", "Madrid", "Rome", "Stockholm"])
products  = np.array(["Laptop", "Phone", "Headphones", "Monitor", "Keyboard",
                      "Mouse", "Tablet", "Printer"])

# --- Default (Silver) lakehouse ---------------------------------------------------
_ctx           = notebookutils.runtime.context
WORKSPACE_ID   = _ctx.get("currentWorkspaceId")   or _ctx.get("workspaceId")
LAKEHOUSE_ID   = _ctx.get("defaultLakehouseId")
LAKEHOUSE_NAME = _ctx.get("defaultLakehouseName") or "default"
if not (WORKSPACE_ID and LAKEHOUSE_ID):
    raise RuntimeError("Attach a default Lakehouse to this notebook before running.")

# delta-rs MUST use ABFSS on Fabric: blobfuse does not support the rename that Delta
# uses for commit files, so local-mount writes fail with
#   "Generic LocalFileSystem error: Unable to rename file (os error 2)".
ABFSS_TABLES_ROOT = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
    f"{LAKEHOUSE_ID}/Tables"
)
STORAGE_OPTIONS = {
    "bearer_token": notebookutils.credentials.getToken("storage"),
    "use_fabric_endpoint": "true",
}

# --- Bronze lakehouse (separate item) --------------------------------------------
# Parquet writes via pyarrow.dataset have no rename step, so a local mount is fine.
# We mount the bronze lakehouse explicitly so Bronze never spills into Silver.
_bronze_info = notebookutils.lakehouse.get(name=BRONZE_LAKEHOUSE_NAME)
_bronze_abfs = _bronze_info.properties["abfsPath"]
_BRONZE_MOUNT_NAME = f"/mnt_{BRONZE_LAKEHOUSE_NAME}"
try:
    notebookutils.fs.mount(_bronze_abfs, _BRONZE_MOUNT_NAME, {"fileCacheTimeout": 0, "timeout": 120})
except Exception:
    # already mounted
    pass
BRONZE_FILES_ROOT = f"{notebookutils.fs.getMountPath(_BRONZE_MOUNT_NAME)}/Files"


StatementMeta(, 224b4d75-d295-42d4-bdc3-7b90622fea45, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 17839724-e840-4f9f-aecc-4e02eeea212f)

In [9]:
_rng = np.random.default_rng()

def chunk_ranges(total, n_chunks):
    """Return list of (start_inclusive, end_exclusive) id ranges, 1..total split across n_chunks."""
    base, rem = divmod(total, n_chunks)
    ranges, cursor = [], 1
    for i in range(n_chunks):
        size = base + (1 if i < rem else 0)
        ranges.append((cursor, cursor + size))
        cursor += size
    return ranges


def build_customers_batch(start, end):
    """Return a pyarrow Table for customer ids in [start, end)."""
    n = end - start
    ids = np.arange(start, end, dtype=np.int64)
    country_idx = _rng.integers(0, len(countries), size=n)
    city_idx    = _rng.integers(0, len(cities),    size=n)
    country = countries[country_idx]
    city    = cities[city_idx]
    # string ops via pandas are fast here because they vectorise over numpy arrays
    names  = pd.Series(ids).astype(str).radd("Customer_")
    emails = names.str.lower() + "@example.com"
    phones = "+1-" + pd.Series(_rng.integers(1_000_000, 10_000_000, size=n)).astype(str)
    return pa.table({
        "id":      ids,
        "name":    names.to_numpy(),
        "country": country,
        "city":    city,
        "email":   emails.to_numpy(),
        "phone":   phones.to_numpy(),
    })


def build_orders_batch(start, end):
    """Return a pyarrow Table for order ids in [start, end), with year/month/day cols."""
    n = end - start
    order_id    = np.arange(start, end, dtype=np.int64)
    customer_id = _rng.integers(1, NUM_CUSTOMERS + 1, size=n, dtype=np.int64)
    product     = products[_rng.integers(0, len(products), size=n)]
    price       = np.round(_rng.random(n) * 495 + 5, 2)
    quantity    = _rng.integers(1, 11, size=n, dtype=np.int32)
    total       = np.round(price * quantity, 2)
    # random order_date in the last 365 days
    today = pd.Timestamp.utcnow().normalize()
    offsets = _rng.integers(0, 365, size=n, dtype=np.int32)
    order_date = (today - pd.to_timedelta(offsets, unit="D")).values.astype("datetime64[D]")
    dates = pd.to_datetime(order_date)
    return pa.table({
        "order_id":    order_id,
        "customer_id": customer_id,
        "product":     product,
        "price":       price,
        "quantity":    quantity,
        "total":       total,
        "order_date":  order_date,
        "year":        dates.year.to_numpy(dtype=np.int32),
        "month":       dates.month.to_numpy(dtype=np.int32),
        "day":         dates.day.to_numpy(dtype=np.int32),
    })


StatementMeta(, 224b4d75-d295-42d4-bdc3-7b90622fea45, 11, Finished, Available, Finished, False)

In [10]:
# Build chunks once, reuse for Bronze (parquet) and Silver (Delta).
# Same pyarrow Tables feed both writers — cheaper than regenerating numpy twice.
CUSTOMER_BATCHES = []  # list of (idx, pa.Table)
ORDER_BATCHES    = []

if STORE_DATA_TO_BRONZE == 1 or STORE_DATA_TO_SILVER == 1:
    for i, (s, e) in enumerate(chunk_ranges(NUM_CUSTOMERS, NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION)):
        CUSTOMER_BATCHES.append((i, build_customers_batch(s, e)))
    for i, (s, e) in enumerate(chunk_ranges(NUM_ORDERS, NUM_ORDER_DELTA_LAKE_TRANSACTION)):
        ORDER_BATCHES.append((i, build_orders_batch(s, e)))
    print(f"Prepared {len(CUSTOMER_BATCHES)} customer batches, {len(ORDER_BATCHES)} order batches")


StatementMeta(, 224b4d75-d295-42d4-bdc3-7b90622fea45, 12, Finished, Available, Finished, False)

Prepared 12 customer batches, 22 order batches


In [11]:
if STORE_DATA_TO_BRONZE == 1:
    from concurrent.futures import ThreadPoolExecutor, as_completed

    print(f"Bronze: writing parquet + hive partitioning to {BRONZE_FILES_ROOT}")

    customers_path = f"{BRONZE_FILES_ROOT}/customers"
    orders_path    = f"{BRONZE_FILES_ROOT}/orders"

    def _write_customers_chunk(idx, table):
        ds.write_dataset(
            table,
            customers_path,
            format="parquet",
            partitioning=ds.partitioning(pa.schema([("country", pa.string())]), flavor="hive"),
            # unique basename per chunk avoids file-name collisions between parallel writers
            basename_template=f"part-customers-{idx:04d}-{{i}}.parquet",
            existing_data_behavior="overwrite_or_ignore",
        )

    def _write_orders_chunk(idx, table):
        ds.write_dataset(
            table,
            orders_path,
            format="parquet",
            partitioning=ds.partitioning(
                pa.schema([("year",  pa.int32()),
                           ("month", pa.int32()),
                           ("day",   pa.int32())]),
                flavor="hive",
            ),
            basename_template=f"part-orders-{idx:04d}-{{i}}.parquet",
            existing_data_behavior="overwrite_or_ignore",
        )

    # Bronze is plain parquet: no Delta transactions, so chunks are independent and
    # safe to run in parallel. Unique basename_template prevents filename collisions
    # between parallel writers into the same hive partition directory.
    with ThreadPoolExecutor(max_workers=NUM_CUSTOMERS_DELTA_LAKE_TRANSACTION) as pool:
        futures = [pool.submit(_write_customers_chunk, idx, t) for idx, t in CUSTOMER_BATCHES]
        for f in as_completed(futures):
            f.result()
    print(f"  -> {customers_path}")

    with ThreadPoolExecutor(max_workers=NUM_ORDER_DELTA_LAKE_TRANSACTION) as pool:
        futures = [pool.submit(_write_orders_chunk, idx, t) for idx, t in ORDER_BATCHES]
        for f in as_completed(futures):
            f.result()
    print(f"  -> {orders_path}")


StatementMeta(, 224b4d75-d295-42d4-bdc3-7b90622fea45, 13, Finished, Available, Finished, False)

Bronze: writing parquet + hive partitioning to /synfs/notebook/224b4d75-d295-42d4-bdc3-7b90622fea45/mnt_bronze/Files
  -> /synfs/notebook/224b4d75-d295-42d4-bdc3-7b90622fea45/mnt_bronze/Files/customers
  -> /synfs/notebook/224b4d75-d295-42d4-bdc3-7b90622fea45/mnt_bronze/Files/orders


In [12]:
def _abfss_rm(path):
    """Recursively remove an abfss path; swallow if it does not exist."""
    try:
        notebookutils.fs.rm(path, True)
    except Exception:
        pass


if STORE_DATA_TO_SILVER == 1:
    print("Silver: writing Delta via delta-rs over ABFSS (N sequential commits for history)")

    customers_delta = f"{ABFSS_TABLES_ROOT}/dbo/customers"
    _abfss_rm(customers_delta)
    # Sequential: one write = one Delta commit, so DESCRIBE HISTORY shows N versions.
    # Reuses the same pyarrow Tables Bronze wrote — no regeneration cost.
    for idx, table in CUSTOMER_BATCHES:
        write_deltalake(
            customers_delta, table,
            mode="append",
            storage_options=STORAGE_OPTIONS,
        )
    print(f"  -> {customers_delta} ({len(CUSTOMER_BATCHES)} commits)")

    orders_delta = f"{ABFSS_TABLES_ROOT}/dbo/orders"
    _abfss_rm(orders_delta)
    for idx, table in ORDER_BATCHES:
        write_deltalake(
            orders_delta, table,
            mode="append",
            partition_by=["year", "month", "day"],
            storage_options=STORAGE_OPTIONS,
        )
    print(f"  -> {orders_delta} ({len(ORDER_BATCHES)} commits)")


StatementMeta(, 224b4d75-d295-42d4-bdc3-7b90622fea45, 14, Finished, Cancelled, Cancelled, False)

Silver: writing Delta via delta-rs over ABFSS (N sequential commits for history)
  -> abfss://0de9654b-6e88-4bd0-a976-428459051081@onelake.dfs.fabric.microsoft.com/cb4ae9ed-24be-4ab6-a398-3a65faf16983/Tables/customers (12 commits)


In [6]:
if STORE_DATA_TO_GOLD == 1:
    import com.microsoft.spark.fabric
    from com.microsoft.spark.fabric.Constants import Constants 
    # Silver Delta tables were written by delta-rs above; Spark auto-discovers them.
    silver_customers = spark.table("silver.dbo.customers")
    silver_orders    = spark.table("silver.dbo.orders")

    # Warehouse: Spark-to-Warehouse connector. Drop first so the write starts clean.
    #spark.sql("DROP TABLE IF EXISTS gold.dbo.gold_customers")
    #spark.sql("DROP TABLE IF EXISTS gold.dbo.gold_orders")

    silver_customers.write.mode("overwrite").synapsesql("gold.dbo.gold_customers")
    silver_orders.write.mode("overwrite").synapsesql("gold.dbo.gold_orders")



StatementMeta(, 224b4d75-d295-42d4-bdc3-7b90622fea45, 8, Finished, Available, Finished, False)

Py4JJavaError: An error occurred while calling o6625.synapsesql.
: com.microsoft.spark.fabric.tds.write.error.FabricSparkTDSWriteError: Write orchestration failed.
	at com.microsoft.spark.fabric.tds.write.processor.FabricTDSWritePreProcessor.orchestrateWrite(FabricTDSWritePreProcessor.scala:66)
	at com.microsoft.spark.fabric.tds.implicits.write.FabricSparkTDSImplicits$FabricSparkTDSWrite.synapsesql(FabricSparkTDSImplicits.scala:84)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractComm